# Assumptions and Inference

As we saw at the beginning of this unit, there are a number of generic assumptions associated with the GLM framework. However, as we established last week, not all these assumptions are relevant and not every metric is interpretable, depending upon the specific choice of GLM. As such, we must first establish *how* these assumptions manifest under Poisson regression, and whether any of the standard assessments remain relevant or useful.


## Poisson Regression Assumption
Under the Poisson  GLM we have been using, the conditional distribution of the outcome variable is

$$
    (y_{i} | x_{i1}, x_{i2}, \dots, x_{ik}) \sim \text{Pois}(\lambda_{i}).
$$

In words, the distribution of $y_{i}$, conditional on the values of all the predictor variables, is assumed to be Poisson with a specific average count $\lambda_{i}$. Thus the three aspects of the GLM become

$$
\begin{align*}
    E(y_{i})   &= \lambda_{i} \\
    \log\left(\lambda_{i}\right) &= \beta_{0} + \beta_{1}x_{i1} + \dots + \beta_{k}x_{ik} \\
    \text{Var}(y_{i}) &= \lambda_{i}
\end{align*}
$$

This then leads to the following interpretations of the assumptions:

1. *The observations are independent of each other (no correlation).*
    - As indicated last week, this remains the case for all GLMs, but is a function of the *design* rather than something we check.
2. *The observations follow the assumed conditional distribution of the outcome variable.*
    - Under this model, the assumption is that the data follow a Poisson distribution of counts for every unique combination of predictor values. This means not only that the distribution is discrete and described by an average count value, but also that the variance of the distribution follows this average count value *exactly*.
3. *The transformed mean is accurately described using the assumed regression model.*
    - This is a key assumption that will need to be checked. On the scale of the link function, we assume the relationship can still be captured by a straight-line and may need to use non-linear methods if this is not the case.
4. *The conditional variance follows the form for the assumed distribution.*
    - This remain true but, unlike the Bernoulli distribution, is a big (and often untenable) assumption that the width of the marginal distribution follows the mean *exactly*. As such, we do not expect homoscedasticity and any diagnostic used to assess constant variance is not useful for this reason. Instead, we assume a very specific form of heteroscedasticity that needs to be checked.

From this we can conclude that the elements we need to check is the accuracy of the regression model on the scale of the link function and whether the Poisson assumption that $E(y_{i}) = \text{Var}(y_{i})$ is reasonable.

### Residual Plots
The standard residual plots produced by `R` generally have a little more utility with Poisson regression than with binary logistic regression. However, some of these plots are also not universally useful. For instance, some of the plots will work if the counts in the dataset are large enough, but for low counts where the distribution is highly skewed, they can be misleading. In addition, none of the plots directly visualise the core assumption of concern, that the mean and variance are *identical*. After the suggestion of [Faraway (2016)](https://www.librarysearch.manchester.ac.uk/permalink/44MAN_INST/1l23p04/alma992979482022301631), we can instead construct a very rough visual assessment of this by plotting the fitted values (i.e. an approximation of the mean) against the squared residuals (i.e. an approximation of the variance) and see whether there is a clear relationship or not. This can also be used to assess whether a quasi-Poisson model could be justifiable, or whether an entirely different form of distribution should be considered. We will see an example of this in the next part of the lesson.

### Plots of the Model Fit
As with logistic regression, we also need to assess the model fit to see whether a straight-line is appropriate on the scale of the link function. For this, we can again use the concept of effects plots and their implementation within the `effects` package. Any clear deviations of the LOESS smoothing line from the fitted regression line are suggestive of non-linear relationships that may require different modelling approaches (such as polynomial regression).

## Hypothesis Tests
Finally, once we are satisfied with the assumptions and model fit, we can turn to inference. Again, this is conducted within the framework of asymptotic Wald tests of the coefficients, or asymptotic likelihood ratio tests of omnibus hypotheses captured by model comparisons.

### Wald Tests
Within the context of Poisson regression, we have to remember that the parameters are on the *log* scale and thus the null value must also be interpretable on this scale. As discussed previously, the intercept represents the log average count when all the predictors are 0. Thus, the traditional null hypothesis

$$
\mathcal{H}_{0}: \beta_{0} = 0
$$

is interpreted as

$$
\log(\lambda_{x=0}) = \beta_{0} = 0,
$$

which we can convert to an actual count with

$$
\lambda_{x=0} = e^{\beta_{0}} = e^{0} = 1.
$$

As such, a hypothesis test of $\mathcal{H}_{0}: \beta_{0} = 0$ is testing the null that the average count when all predictors are 0 is 1. This is the meaning of the test on the intercept.

```{tip} Why Not a Count of 0?
You might think that it would make more sense to test against a count of 0, rather than 1. Unfortunately, this has exactly the same problems with testing a probability of 0 in logistic regression. Firstly, the null tests are generic across all GLMs, including those that do not model probability or counts, making a null value of 0 on the scale of the link function the simplest choice. The second problem is that you cannot take the log of 0. So while $\lambda_{x=0} = 0$ might make intuitive sense, this would be $\log 0 = -\infty$ on the scale of the link function. This also highlights why Poisson regression will never predict an average count of exactly 0. It can get trivially close to 0, but never exactly 0, due to the link function.
```

For the slope, we need to remember the interpretation that this represents the log of a *count-ratio*, reflecting how the count changes for each unit increase in the predictor. If the relationship were a flat line then the count would not change as $x$ increases. For instance, if the average count was 5 for both $x = 0$ and $x = 1$. This would lead to

$$
e^{\beta_{j}} = \frac{5}{5} = 1.
$$

Converting this back to the scale of $\beta_{j}$, we get

$$
\beta_{j} = \log(1) = 0.
$$

As such, a hypothesis test of $\mathcal{H}_{0}: \beta_{j} = 0$ is testing the null that the average count *does not change* as $x$ increases. This is a *flat line* on the log scale and is therefore the meaning of the tests on the slopes. Note that this is *not* saying that the average count is 0. It is saying that the average count, whatever it is, does not change.

We can also think of this in terms of multiplicative changes in the count. As we saw in the previous part of the lesson, if the expected count at $x=0$ is 10 then a null of $\beta_{1} = 0$ gives $e^{\beta_{1}} = 1$ and we have

$$
\lambda_{x=1} = 10 \times 1 = 10
$$

and the count does not change. This is equivalent to a percentage change of $(e^{\beta_{1}} - 1) \times 100 = 0\%$. So, either way you look at it, the null hypothesis is that the average count does not change with increasing values of $x$.

### Analysis of Deviance
In addition to the coefficient tests, we can also create Analysis of Deviance tables for omnibus tests of groups of coefficients ... HERE

In order to develop an omnibus tests of *current correction*, we compare the following null ($\mathcal{M}_{0}$) and full ($\mathcal{M}_{1}$) models:

$$
\begin{align*}
    \mathcal{M}_{0} &: \log\left(\frac{\pi_{i}}{1 - \pi_{i}}\right) = \beta_{0} \\
    \mathcal{M}_{1} &: \log\left(\frac{\pi_{i}}{1 - \pi_{i}}\right) = \beta_{0} + \left(\beta_{1} \times \text{CORR1}_{i}\right) + \left(\beta_{2} \times \text{CORR2}_{i}\right) \\
\end{align*}
$$

In other words, a model that contains the factor *current correction* and a model that does not. If the inclusion of this factor provides a better fit to the data, then we conclude that there is some meaningful effect of *current correction* that explains the variation in the data. This is the same logic as the traditional ANOVA. The only difference is that, in a GLM, this comparison is one of *deviance* not *variance*, and is conducted using a likelihood ratio test and its associated asymptotic $\chi^{2}$ statistic. Everything else remains exactly the same. As such, everything you already known about the ANOVA can be directly applied to a binary outcome.

:::{tip .simple icon=false} What You Now Know
In this section, we have discussed both assumptions and inference for binary logistic regression. After reading this section, you should have a good sense of:
- The idea that the only assumption we need to assess for binary logistic regression is the model fit, inclusive of:
    - The linearity of the effects on the scale of the link function.
    - The absence of extreme values that are influencing the fit.
- The idea that standard residuals plots may not be that useful for binary logistic regression.
- The idea that the most useful method for assessing the model fit is through the use of *effects plots*, which show a single effect in the model with all others held at their average value.
- The idea that hypothesis tests can be performed either as Wald tests of the individual regression coefficients or as omnibus tests, using the likelihood ratio (LR).
- For the Wald tests, the null value remains 0 on the scale of the link function. This means:
    - For the intercept, that the test is one of 50% probability at baseline.
    - For the slope, that the test is one of probability not changing as a function of the predictor.
- For the LR tests, the null value concerns no change in deviance between two models. This is effectively the same as conceptualising an ANOVA in terms of change in error variance. As such, we can perform an *Anaysis of Deviance* as a direct analogue to the traditional ANOVA.
:::